# Amendment A2: chạy trên GPU thuê (Linux Docker)

Notebook này điều khiển các bước của chặng II trong `docs/AMENDMENT_A2_RUNBOOK_VI.md`. Mỗi ô gọi Python của
`.venv` trong repo qua `subprocess` và tự đặt `CUBLAS_WORKSPACE_CONFIG`, nên chạy được với kernel bất kỳ (kể cả
Python 3.10 của hệ thống).

**Đặt notebook ngoài repo** (ví dụ `/workspace/a2_gpu_run.ipynb`), tải bằng:

```
wget -O /workspace/a2_gpu_run.ipynb https://raw.githubusercontent.com/quanuiter/SleepTCN/main/notebooks/a2_gpu_run.ipynb
```

Không `git pull`/`checkout` trong repo khi chiến dịch đang chạy.

Mọi ô đều chạy lại được: bước đã xong được bỏ qua, bước dở được tiếp tục. Mỗi ô dừng với `RuntimeError` nếu kết quả
không đúng; khi đó gửi vài dòng cuối của output để xử lý, **không xóa thư mục nào**.

Thứ tự: ô 1 (thiết lập) → ô 2 (trạng thái) → nếu train chưa chạy thì ô 3 → **ô 4 chạy toàn bộ phần còn lại**.
Các ô 5–9 là từng bước riêng lẻ, dùng khi cần chạy tay.

In [ ]:
# Ô 1. Thiết lập (chạy đầu tiên, mỗi lần mở lại notebook)
import json, os, subprocess, time, hashlib
from pathlib import Path

REPO = Path("/workspace/SleepTCN")
PY = str(REPO / ".venv" / "bin" / "python")
ENV = {**os.environ, "CUBLAS_WORKSPACE_CONFIG": ":4096:8", "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1"}
MON = REPO / "runs" / "a2" / "monitoring"
TRAIN_WORKERS, TRAIN_LOADERS = 3, 1      # 6-core V100 machine (runbook II.7)
HOLDOUT_WORKERS, HOLDOUT_THREADS = 2, 2
EXPORT = Path("/workspace/a2_results.tar.gz")


def run(args, expect=None, cwd=REPO):
    """Run a command with the venv environment, stream its output, check exit code and expected text."""
    print("$", " ".join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=cwd, env=ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    output = "".join(lines)
    if process.wait() != 0:
        raise RuntimeError(f"command failed (exit {process.returncode}): {' '.join(map(str, args))}")
    if expect and expect not in output:
        raise RuntimeError(f"expected {expect!r} in the output of {' '.join(map(str, args))}")
    return output


def a2(*args, expect=None):
    return run([PY, "scripts/run_amendment_a2.py", *args], expect=expect)


def states():
    """Counts of A2 training states plus the reproduction-check state (reads files only)."""
    snippet = (
        "import json,sys; sys.path.insert(0,'src'); from pathlib import Path; "
        "from sleeptcn.amendment_a2 import load_a2_config,training_targets,training_state,repro_target; "
        "w=Path('.').resolve(); c,_=load_a2_config(w); n={}; "
        "[n.__setitem__(s,n.get(s,0)+1) for s in (training_state(w,t) for t in training_targets(c))]; "
        "print(json.dumps({'runs':n,'repro':training_state(w,repro_target())}))")
    result = subprocess.run([PY, "-c", snippet], cwd=REPO, env=ENV, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stdout + result.stderr)
    return json.loads(result.stdout.strip().splitlines()[-1])


def training_running():
    """True while the 'train' driver or a 'train-worker' process of this repository is alive."""
    listing = subprocess.run(["ps", "-eo", "args"], capture_output=True, text=True).stdout
    return any("run_amendment_a2.py" in line and (" train " in line + " " or "train-worker" in line)
               for line in listing.splitlines())


def tail(path, n=5):
    path = Path(path)
    if path.is_file():
        print(f"--- {path.name}")
        print("".join(path.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)[-n:]))


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def start_training():
    s = states()
    if s["repro"] not in ("trained", "tested"):
        raise RuntimeError("run the reproduction check first (ô 5)")
    if s["runs"].get("trained", 0) + s["runs"].get("tested", 0) == 50:
        print("all 50 runs already trained"); return
    if training_running():
        print("training is already running"); return
    MON.mkdir(parents=True, exist_ok=True)
    log = open(MON / "train_driver.log", "a", encoding="utf-8")
    subprocess.Popen([PY, "scripts/run_amendment_a2.py", "train", "--device", "cuda",
                      "--workers", str(TRAIN_WORKERS), "--num-workers", str(TRAIN_LOADERS)],
                     cwd=REPO, env=ENV, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(60)
    if not training_running():
        tail(MON / "train_driver.log", 30)
        raise RuntimeError("training stopped right after start; see train_driver.log above")
    print("training started in the background")


def wait_for_training(poll_minutes=5):
    while True:
        s = states()
        done = s["runs"].get("trained", 0) + s["runs"].get("tested", 0)
        print(time.strftime("%H:%M"), "trained:", done, "/ 50", s["runs"], flush=True)
        if done == 50:
            return
        if not training_running():
            for i in range(TRAIN_WORKERS):
                tail(MON / f"train_worker_{i}.log", 15)
            tail(MON / "train_driver.log", 15)
            raise RuntimeError("training is not running but not all 50 runs are trained; "
                               "if the logs show no real error, run ô 3 to resume, then ô 4 again")
        time.sleep(poll_minutes * 60)


def test_campaign():
    journal = REPO / "runs" / "a2" / "test_campaign_a2.json"
    if journal.is_file() and json.loads(journal.read_text())["status"] == "complete":
        print("test campaign already complete"); return
    if not journal.is_file():  # not started yet: read-only preflight before opening the test folds
        a2("test", expect="Preflight passed for 50 runs")
    a2("test", "--device", "cuda", "--confirm", "OPEN-A2-LOCKED-TEST-ONCE", expect="A2 TEST CAMPAIGN: complete")


def holdout():
    if (REPO / "runs" / "a2" / "holdout" / "run_manifest.json").is_file():
        print("holdout inference already complete"); return
    a2("holdout-check", "--device", "cuda", expect="DEVICE CHECK PASSED")
    a2("holdout", "--device", "cuda", "--workers", str(HOLDOUT_WORKERS), "--threads", str(HOLDOUT_THREADS),
       expect="A2 HOLDOUT INFERENCE COMPLETE")


def step_benchmark():
    if (REPO / "runs" / "a2_step_time_benchmark.json").is_file():
        print("step-time benchmark already done"); return
    if training_running():
        raise RuntimeError("another training process is still running; the benchmark must run alone")
    a2("step-benchmark", "--device", "cuda")


def export():
    a2("export", "--output", EXPORT, expect="EXPORTED")
    digest = sha256(EXPORT)
    (EXPORT.parent / (EXPORT.name + ".sha256")).write_text(f"{digest}  {EXPORT.name}\n")
    print(f"\nDOWNLOAD {EXPORT} ({EXPORT.stat().st_size / 1e9:.2f} GB)\nSHA-256 {digest}")


assert Path(PY).is_file(), f"{PY} not found: create the venv first (runbook II.2)"
a2("env-check", "--device", "cuda", expect="ENVIRONMENT OK")
print("SETUP OK")

In [ ]:
# Ô 2. Trạng thái (chạy bất cứ lúc nào)
s = states()
print("training states:", s["runs"], "| reproduction check:", s["repro"])
print("training process running:", training_running())
for i in range(TRAIN_WORKERS):
    tail(MON / f"train_worker_{i}.log", 2)
for name in ("test_campaign_a2.json", "holdout/run_manifest.json"):
    print(name, "exists:", (REPO / "runs" / "a2" / name).is_file())
print("step benchmark exists:", (REPO / "runs" / "a2_step_time_benchmark.json").is_file())
print(subprocess.run(["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total",
                      "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# Ô 3. Bắt đầu hoặc tiếp tục train chạy nền (bỏ qua nếu đang chạy hoặc đã xong)
start_training()

In [ ]:
# Ô 4. CHẠY TOÀN BỘ PHẦN CÒN LẠI: chờ train xong -> mở test -> holdout -> đo thời gian -> xuất kết quả.
# Dừng ngay khi có lỗi. Chạy lại sau khi sửa: các bước đã xong được bỏ qua.
wait_for_training()
test_campaign()
holdout()
step_benchmark()
export()
print("\nALL GPU STEPS DONE: download the archive, check its SHA-256, then stop the container.")

## Các bước riêng lẻ (chỉ dùng khi cần chạy tay)

In [ ]:
# Ô 5. Kiểm tra tái lập (đã chạy rồi thì bỏ qua)
if states()["repro"] in ("trained", "tested"):
    print("reproduction check already done:"); print((REPO / "runs/a2/reproduction_check.json").read_text()[:2000])
else:
    a2("repro", "--device", "cuda", expect="REPRODUCTION CHECK")

In [ ]:
# Ô 6. Chỉ chờ train xong
wait_for_training()

In [ ]:
# Ô 7. Chỉ mở tập test Sleep-EDF (một lần duy nhất)
test_campaign()

In [ ]:
# Ô 8. Chỉ suy luận holdout
holdout()

In [ ]:
# Ô 9. Chỉ đo thời gian mỗi bước, rồi xuất kết quả
step_benchmark()
export()